In [ ]:
from airbot_data_collection.common.datasets.mcap_dataset import (
    McapFlatbufferSampleDataset,
    McapDatasetConfig,
)
from typing import Dict
from pydantic import BaseModel
from collections import defaultdict
from pathlib import Path
from PIL import Image
from pydantic_settings import CliApp
from pprint import pprint
from IPython.display import display, clear_output
from airbot_data_collection.utils import resolution_to_inches


class Config(BaseModel):
    """Configuration for the MCAP dataset viewer.
    Args:
        data_root (str): Path to the MCAP file or directory containing MCAP files.
        topics (list[str]): List of topics to load from the MCAP files.
    """

    data_root: str
    topics: Dict[str, Dict[int, str]]
    output_path: str = "frames"


# config = CliApp.run(Config)
config = Config(
    data_root="/home/ghz/下载/emergency_button.mcap",
    topics={
        # "/lead/arm/pose/position": {
        #     # 0: "position_x (m)",
        #     # 1: "position_y (m)",
        #     2: "position_z (m)",
        # },
        # TODO: support common config, e.g. unit
        "/lead/arm/wrench/force": {
            # 0: "force_x (N)",
            # 1: "force_y (N)",
            2: "force_z (N)",
        },
    },
)

Path(config.output_path).mkdir(parents=False, exist_ok=True)

dataset = McapFlatbufferSampleDataset(
    McapDatasetConfig(
        data_root=config.data_root,
        topics=config.topics.keys(),
    )
)
dataset.load()

# for index, sample in enumerate(dataset.reader.iter_attachment_samples(color_topics)):
#     # print(f"Sample {index}: {sample.keys()}")
#     # print(index)
#     pass

import holoviews as hv
from tqdm import tqdm
import time

# backend = "bokeh"  # "matplotlib"  # "bokeh"
backend = "matplotlib"  # "matplotlib"  # "bokeh"

hv.extension(backend)
hv.output(max_frames=600)  # 设置最大帧数以避免警告

diff_cfg = {
    "matplotlib": {
        "s": 50,
    },
    "bokeh": {"size": 8},
}

frames = defaultdict(lambda: defaultdict(dict))
xs = []
# xs = defaultdict(list)
ys = defaultdict(lambda: defaultdict(list))
image_size = None
for i, sample in tqdm(enumerate(dataset), total=len(dataset)):
    if i == 100:
        break
    for key, value in sample.items():
        for index, name in config.topics[key].items():
            cur_ys = ys[key][index]
            field_path = f"{key.removeprefix('/').replace('/', '.')}.{name}"
            if i == 0:
                # 第 0 帧：空白 (空 Overlay)
                curve = hv.Curve([])
                point = hv.Scatter([])
            elif i == 1:
                # 第 1 帧：只有第一个点
                curve = hv.Curve([])
                point = hv.Scatter((xs, cur_ys)).opts(**diff_cfg[backend], color="red")
            else:  # >=2
                # 第 2 帧开始：曲线 + 笔头点
                label = "End-effector Force Trajectory"
                # label = label or field_path
                label = ""
                # kdims = "t"
                kdims = "t (a.u.)"
                curve = hv.Curve((xs, cur_ys), kdims, name, label=label).opts(
                    linewidth=3
                )
                point = hv.Scatter(
                    (xs[i - 1 : i], cur_ys[i - 1 : i]), kdims, name, label=label
                ).opts(**diff_cfg[backend], color="red")
            frame = curve * point
            # frame.opts(xlim=(0, len(dataset)))
            # frame.opts(ylim=(3.28, 19.9))
            if backend == "matplotlib":
                frame.opts(fig_inches=(25, 12), aspect="auto")
                frame.opts(
                    fontsize={
                        "xlabel": 25,
                        "ylabel": 25,
                        "xticks": 20,
                        "yticks": 20,
                        "title": 20,
                    }
                )
            else:
                frame.opts(width=250, height=250)

            # image_path = (
            #     f"frames/{key.removeprefix('/').replace('/', '.')}.{name}.{i}.png"
            # )
            # hv.save(frame, image_path, fmt="png", backend=backend, dpi=110)
            # img = Image.open(image_path)
            # if image_size is None:
            #     image_path = f"frames/{field_path}.{i}.png"
            #     hv.save(frame, image_path, fmt="png", backend=backend, dpi=110)
            #     img = Image.open(image_path)
            #     assert img.size[0] % 2 == 0 and img.size[1] % 2 == 0, (
            #         f"图像尺寸必须为偶数，但得到 {img.size}"
            #     )
            #     image_size = img.size
            # else:
            #     assert img.size == image_size, (
            #         f"图像尺寸不一致，之前为 {image_size}，但得到 {img.size}"
            #         f"{image_path}"
            #     )
            frames[key][index][i] = frame
            # FIXME: 支持为不同的value的index配置overlay或者layout
            ys[key][index].append(sample[key][index])
            # display(frame)
            # time.sleep(0.05)
            # input("Press Enter to continue...")
            # clear_output(wait=True)
    xs.append(i)

frame

# stat = {}
# for key, value in ys.items():
#     for index, traj in value.items():
#         stat[f"{key}_{index}"] = (min(traj), max(traj))
# pprint(stat)

# print("Creating HoloMap...")
# holomap = hv.HoloMap(frames[key][2], kdims="Time")
# print("Setting options...")
# # holomap.opts(width=600, height=400, show_frame=True, title="Sine Wave Animation", xlim=(0, None))

# # 保存为 mp4 (需要 ffmpeg)
# print("Saving to mp4...")
# hv.save(holomap, "sine_grow_from0", fmt="mp4")

In [ ]:
!ffmpeg -framerate 25 -y -i "frames/lead.arm.wrench.force.force_z.%d.png" -c:v libx264 -pix_fmt yuv420p video.mp4